# ruBERT Iteration 2 — BERT-2 vs BERT-3 (s2l augmentation)

BERT-0 and BERT-1 (previous notebook) reached high token-F1 (~0.83) but low
span-F1 (~0.19) on the real Gosha test set:

- BERT-0 (185 synthetic): Test Span-F1 = 0.1932±0.0043
- BERT-1 (curriculum 1261 Mathematicon → 185 synthetic): Test Span-F1 = 0.1844±0.0177

Root cause: span-length mismatch. Synthetic spans average 10.0 tokens (0%
single-token, 51% ≥10); Mathematicon spans average 2.8 (52% single-token);
the real test set (Gosha) averages 6.8 (25% single-token, 25% ≥10).

This notebook augments training data with ~2000 s2l-derived examples using
**two methods**:

- **Method A — template injection** (~1480 examples): extract a carrier
  template from each of the 185 `train_clean` sentences (math spans →
  `{SLOT_k}` placeholders), then fill slots with s2l pronunciations to
  produce 8 variants per template.
- **Method B — simple carrier phrases** (~520 examples): wrap s2l
  pronunciations directly in 16 short carrier phrases for extra variety.

s2l sampling is stratified by pronunciation token length (20% 3-5, 30% 6-9,
30% 10-15, 20% 16-25) to match the real test distribution.

Two new experiments, each over 3 seeds (42, 123, 456):

- **BERT-2** — `train_clean` (185) + s2l_augmented (~2000), no curriculum.
- **BERT-3** — curriculum: Stage 1 on `train_noisy` (Mathematicon), Stage 2
  on `train_clean` + s2l_augmented, fresh optimizer for Stage 2.

**Performance note:** in the previous run, writing model checkpoints to
disk ("Writing model shards...") accounted for ~90-95% of total wall-clock
training time. This notebook avoids that entirely for stages that evaluate:
instead of `save_strategy="epoch"` + `load_best_model_at_end`, a
`BestModelTracker` callback keeps the best-eval weights in CPU memory and
restores them at the end of training, with zero disk checkpoint writes
during the loop. Early stopping is reproduced via the same
patience-counts-bad-evals logic.

The final cell prints a single `<executor_output id="E4b">` block. BERT-0 /
BERT-1 are **not** rerun — their numbers above are hardcoded into the final
comparison table.

**Before running:** upload/mount the exported HF `DatasetDict` and the s2l
CSV (`russian_sentences_train.csv`, columns `pronunciation`,
`sentence_normalized`), and set `HF_DATA_PATH` / `S2L_CSV_PATH` below.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/linguistic project/hf_dataset.zip" -d "/content/hf_dataset"

Archive:  /content/drive/MyDrive/linguistic project/hf_dataset.zip
   creating: /content/hf_dataset/hf_dataset/
  inflating: /content/hf_dataset/hf_dataset/dataset_dict.json  
   creating: /content/hf_dataset/hf_dataset/test/
   creating: /content/hf_dataset/hf_dataset/train_clean/
   creating: /content/hf_dataset/hf_dataset/train_noisy/
   creating: /content/hf_dataset/hf_dataset/val/
  inflating: /content/hf_dataset/hf_dataset/test/state.json  
  inflating: /content/hf_dataset/hf_dataset/test/dataset_info.json  
  inflating: /content/hf_dataset/hf_dataset/test/data-00000-of-00001.arrow  
  inflating: /content/hf_dataset/hf_dataset/train_clean/state.json  
  inflating: /content/hf_dataset/hf_dataset/train_clean/dataset_info.json  
  inflating: /content/hf_dataset/hf_dataset/train_clean/data-00000-of-00001.arrow  
  inflating: /content/hf_dataset/hf_dataset/train_noisy/state.json  
  inflating: /content/hf_dataset/hf_dataset/train_noisy/dataset_info.json  
  inflating: /content/hf_data

In [9]:
!cp "/content/drive/MyDrive/linguistic project/russian_sentences_train.csv" "/content/"

In [4]:
# Colab ships a `peft` build that's ahead of older `accelerate`/`transformers`
# pins (it imports `accelerate.utils.memory.clear_device_cache`, added later),
# which breaks `from transformers import Trainer` with an ImportError about
# peft/accelerate. We don't use peft/LoRA anywhere in this notebook, so the
# simplest fix is to upgrade transformers/accelerate together and drop peft.
!pip uninstall -y -q peft
!pip install -q -U "transformers>=4.42" "accelerate>=0.31" "datasets>=2.19" "seqeval==1.2.2" scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 100.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 122.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 122.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12

In [5]:
import os, gc, time, inspect, shutil, warnings, re, random

import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, Dataset, concatenate_datasets, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForTokenClassification,
    set_seed,
)
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.scheme import IOB2
from sklearn.metrics import f1_score as sklearn_f1

warnings.filterwarnings("ignore")

# ---- Config ----
MODEL_NAME = "ai-forever/ruBert-base"
FALLBACK_MODEL_NAME = "DeepPavlov/rubert-base-cased"  # escape hatch only, flagged if used
HF_DATA_PATH = "./hf_dataset/hf_dataset"                    # <-- set to wherever the DatasetDict was uploaded/mounted
S2L_CSV_PATH = "./russian_sentences_train.csv"      # <-- set to wherever the s2l CSV was uploaded
MAX_LENGTH = 128
SEEDS = [42, 123, 456]
AUG_SEED = 42

LABEL_LIST = ["O", "B-MATH", "I-MATH"]
ID2LABEL = {0: "O", 1: "B-MATH", 2: "I-MATH"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(LABEL_LIST)

# Stratified length buckets shared by both augmentation methods: (min, max, weight).
LENGTH_BUCKETS = [(3, 5, 0.20), (6, 9, 0.30), (10, 15, 0.30), (16, 25, 0.20)]

METHOD_A_VARIANTS_PRIMARY = 8
METHOD_A_VARIANTS_FALLBACK = 6   # escape hatch: used if total skips > threshold
METHOD_B_TOTAL = 520
AUG_SKIP_RESAMPLE_THRESHOLD = 200

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Training will be extremely slow / may exceed Colab CPU limits.")

# Flags collected during the run, surfaced verbatim in the final Notes section.
RUN_FLAGS = []

def flag(msg):
    print(f"[FLAG] {msg}")
    RUN_FLAGS.append(msg)

Device: cuda


## 1. Load tokenizer

In [6]:
def load_tokenizer_and_model_name():
    global MODEL_NAME
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        return tok, MODEL_NAME
    except Exception as e1:
        flag(f"First attempt to load tokenizer for {MODEL_NAME} failed: {e1}. Retrying with trust_remote_code=True.")
        try:
            tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
            return tok, MODEL_NAME
        except Exception as e2:
            flag(
                f"Second attempt for {MODEL_NAME} failed: {e2}. "
                f"Falling back to {FALLBACK_MODEL_NAME} -- RESULTS BELOW USE THE FALLBACK MODEL, NOT ruBert-base."
            )
            tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
            return tok, FALLBACK_MODEL_NAME


tokenizer, ACTIVE_MODEL_NAME = load_tokenizer_and_model_name()
print(f"Active model checkpoint: {ACTIVE_MODEL_NAME}")

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

Active model checkpoint: ai-forever/ruBert-base


## 2. Load dataset

In [7]:
assert os.path.exists(HF_DATA_PATH), (
    f"HF_DATA_PATH '{HF_DATA_PATH}' not found. Upload/mount the exported hf_dataset "
    f"folder and update HF_DATA_PATH above before re-running this cell."
)

raw_datasets = load_from_disk(HF_DATA_PATH)
assert isinstance(raw_datasets, DatasetDict), "Expected a DatasetDict at HF_DATA_PATH"

for split in ["train_clean", "train_noisy", "val", "test"]:
    assert split in raw_datasets, f"Missing required split '{split}' in dataset"

for split, ds in raw_datasets.items():
    n_tok = sum(len(ex["tokens"]) for ex in ds)
    print(f"{split:12s} n_sentences={len(ds):5d}  n_tokens={n_tok:6d}  avg_len={n_tok/len(ds):.1f}")

print("\nSample example (train_clean[0]):")
print(raw_datasets["train_clean"][0])


def length_filter_dataset(ds, max_length, name):
    '''Per spec: any TRAINING sentence whose tokenization exceeds max_length
    subwords is SKIPPED (not truncated), with the skip count printed/flagged.
    Only applied to train_clean/train_noisy here -- val/test are intentionally
    left untouched (composition must not change); they get truncated, not
    dropped, inside tokenize_and_align_labels if ever needed.'''
    keep_indices, n_skip = [], 0
    for i, ex in enumerate(ds):
        enc = tokenizer(ex["tokens"], is_split_into_words=True, truncation=False)
        if len(enc["input_ids"]) > max_length:
            n_skip += 1
        else:
            keep_indices.append(i)
    if n_skip > 0:
        flag(f"{name}: skipped {n_skip} sentence(s) exceeding {max_length} subword tokens "
             f"(out of {len(ds)}) before training.")
    print(f"{name}: {len(ds)} total, {n_skip} skipped for length (>{max_length} subwords), "
          f"{len(keep_indices)} kept for training.")
    return ds.select(keep_indices)


# Apply the skip (not truncate) rule to both original training splits. In practice
# these are short transcript sentences and 0 should be skipped, but this keeps the
# notebook correct even if that assumption ever breaks.
raw_datasets["train_clean"] = length_filter_dataset(raw_datasets["train_clean"], MAX_LENGTH, "train_clean")
raw_datasets["train_noisy"] = length_filter_dataset(raw_datasets["train_noisy"], MAX_LENGTH, "train_noisy")

train_clean  n_sentences=  185  n_tokens=  6259  avg_len=33.8
train_noisy  n_sentences= 1261  n_tokens= 21477  avg_len=17.0
val          n_sentences=   30  n_tokens=  1105  avg_len=36.8
test         n_sentences=   50  n_tokens=  2031  avg_len=40.6

Sample example (train_clean[0]):
{'sentence_id': 'synth_0000', 'tokens': ['По', 'свойству', 'логарифмов,', 'логарифм', 'произведения', 'экспоненты', 'в', 'степени', 'квадрата', 'логарифма', 'икс', 'и', 'игрек', 'распадается', 'в', 'сумму,', 'а', 'логарифм', 'от', 'произведения', 'а', 'и', 'бэ', 'равен', 'сумме', 'логарифмов', 'сомножителей.'], 'ner_tags': [0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 0, 0, 0, 0]}
train_clean: 185 total, 0 skipped for length (>128 subwords), 185 kept for training.
train_noisy: 1261 total, 0 skipped for length (>128 subwords), 1261 kept for training.


## 3. Load s2l data & build templates

In [10]:
assert os.path.exists(S2L_CSV_PATH), (
    f"S2L_CSV_PATH '{S2L_CSV_PATH}' not found. Upload/mount russian_sentences_train.csv "
    f"and update S2L_CSV_PATH above before re-running this cell."
)

s2l_df = pd.read_csv(S2L_CSV_PATH)
assert "pronunciation" in s2l_df.columns, f"Expected a 'pronunciation' column, got {list(s2l_df.columns)}"
n_s2l_loaded = len(s2l_df)
print(f"s2l rows loaded: {n_s2l_loaded}")

s2l_df["pronunciation"] = s2l_df["pronunciation"].astype(str)
s2l_df["n_tokens"] = s2l_df["pronunciation"].str.split().apply(len)
filtered_s2l = s2l_df[(s2l_df["n_tokens"] >= 3) & (s2l_df["n_tokens"] <= 25)].reset_index(drop=True)
n_s2l_filtered = len(filtered_s2l)
print(f"s2l rows after filtering to 3-25 tokens: {n_s2l_filtered}")


def make_bucket_pools(df, buckets):
    pools = []
    for lo, hi, w in buckets:
        pool = df[(df["n_tokens"] >= lo) & (df["n_tokens"] <= hi)]["pronunciation"].tolist()
        if len(pool) == 0:
            flag(f"s2l bucket {lo}-{hi} tokens is empty after filtering; sampling will skip this bucket.")
        pools.append(pool)
    return pools


_BUCKET_POOLS = make_bucket_pools(filtered_s2l, LENGTH_BUCKETS)
_BUCKET_WEIGHTS = [w for (_, _, w) in LENGTH_BUCKETS]


def sample_pronunciation(rng):
    """Pick a length bucket according to LENGTH_BUCKETS weights (renormalized
    over non-empty buckets), then a pronunciation uniformly at random from it."""
    avail = [(p, w) for p, w in zip(_BUCKET_POOLS, _BUCKET_WEIGHTS) if len(p) > 0]
    assert avail, "No non-empty s2l length buckets available -- cannot sample."
    pools, weights = zip(*avail)
    chosen_pool = rng.choices(pools, weights=weights, k=1)[0]
    return rng.choice(chosen_pool)


_TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def simple_tokenize(text):
    """Whitespace+punctuation tokenizer for carrier prefix/suffix text.
    Pronunciation tokens themselves are already whitespace-separated words,
    so those are split with plain .split()."""
    return _TOKEN_RE.findall(text) if text else []


def extract_template(tokens, ner_tags):
    """Replace each contiguous B-MATH(I-MATH)* span with a ('__SLOT__', k)
    sentinel; keep O tokens as literal strings. Returns (items, n_slots)."""
    items = []
    n_slots = 0
    i, n = 0, len(tokens)
    while i < n:
        if ner_tags[i] == 1:
            j = i + 1
            while j < n and ner_tags[j] == 2:
                j += 1
            items.append(("__SLOT__", n_slots))
            n_slots += 1
            i = j
        else:
            items.append(tokens[i])
            i += 1
    return items, n_slots


templates = []
for ex in raw_datasets["train_clean"]:
    items, n_slots = extract_template(ex["tokens"], ex["ner_tags"])
    assert n_slots >= 1, f"train_clean example {ex.get('sentence_id', '?')} has no math spans"
    templates.append({"items": items, "n_slots": n_slots})

slot_counts = sorted(set(t["n_slots"] for t in templates))
print(f"Extracted {len(templates)} templates from train_clean (distinct slot counts: {slot_counts})")
print("Sample template items:", templates[0]["items"])

s2l rows loaded: 71108
s2l rows after filtering to 3-25 tokens: 59954
Extracted 185 templates from train_clean (distinct slot counts: [2, 3])
Sample template items: ['По', 'свойству', 'логарифмов,', ('__SLOT__', 0), 'распадается', 'в', 'сумму,', 'а', ('__SLOT__', 1), 'равен', 'сумме', 'логарифмов', 'сомножителей.']


## 4. Augment: Method A (template injection) + Method B (carrier phrases)

In [11]:
def build_method_a(templates, variants_per_template, seed):
    rng = random.Random(seed)
    examples = []
    for t_idx, tmpl in enumerate(templates):
        for _ in range(variants_per_template):
            tokens, ner_tags = [], []
            for item in tmpl["items"]:
                if isinstance(item, tuple) and item[0] == "__SLOT__":
                    pron_tokens = sample_pronunciation(rng).split()
                    tokens.extend(pron_tokens)
                    ner_tags.extend([1] + [2] * (len(pron_tokens) - 1))
                else:
                    tokens.append(item)
                    ner_tags.append(0)
            examples.append({
                "sentence_id": f"s2l_tmpl_{len(examples):04d}",
                "tokens": tokens,
                "ner_tags": ner_tags,
                "span_lengths": [
                    sum(1 for t in ner_tags if t in (1, 2))  # placeholder, recomputed below
                ],
            })
    # Recompute true per-span lengths (a sentence can have multiple spans).
    for e in examples:
        lens, cur = [], 0
        for t in e["ner_tags"]:
            if t == 1:
                if cur:
                    lens.append(cur)
                cur = 1
            elif t == 2:
                cur += 1
            else:
                if cur:
                    lens.append(cur)
                cur = 0
        if cur:
            lens.append(cur)
        e["span_lengths"] = lens
    return examples


SIMPLE_CARRIERS = [
    ("Рассмотрим", "{MATH}", "."),
    ("Получаем, что", "{MATH}", "."),
    ("Вычислим", "{MATH}", "."),
    ("Пусть", "{MATH}", "."),
    ("По определению,", "{MATH}", "."),
    ("Значит,", "{MATH}", "."),
    ("Следовательно,", "{MATH}", "."),
    ("Запишем", "{MATH}", "."),
    ("Обозначим", "{MATH}", "."),
    ("Имеем", "{MATH}", "."),
    ("Выражение", "{MATH}", "является решением ."),
    ("Если", "{MATH}", ", то утверждение доказано ."),
    ("Так как", "{MATH}", ", то имеем противоречие ."),
    ("Из равенства", "{MATH}", "следует утверждение теоремы ."),
    ("Проверим, что", "{MATH}", "удовлетворяет условию ."),
    ("{MATH}", ".", ""),
]


def carrier_to_prefix_suffix(carrier_tuple):
    """Locate the literal '{MATH}' marker inside the tuple; everything before
    it joins into the prefix, everything after into the suffix. Handles the
    bare-formula carrier ("{MATH}", ".", "") where the marker is element 0."""
    idx = carrier_tuple.index("{MATH}")
    prefix = " ".join(carrier_tuple[:idx]).strip()
    suffix = " ".join(carrier_tuple[idx + 1:]).strip()
    return prefix, suffix


def build_method_b(carriers, total, seed):
    rng = random.Random(seed)
    n_carriers = len(carriers)
    base, remainder = divmod(total, n_carriers)
    counts = [base + (1 if i < remainder else 0) for i in range(n_carriers)]

    examples = []
    for carrier, n in zip(carriers, counts):
        prefix, suffix = carrier_to_prefix_suffix(carrier)
        prefix_tokens = simple_tokenize(prefix)
        suffix_tokens = simple_tokenize(suffix)
        for _ in range(n):
            pron_tokens = sample_pronunciation(rng).split()
            tokens = prefix_tokens + pron_tokens + suffix_tokens
            ner_tags = [0] * len(prefix_tokens) + [1] + [2] * (len(pron_tokens) - 1) + [0] * len(suffix_tokens)
            examples.append({
                "sentence_id": f"s2l_carr_{len(examples):04d}",
                "tokens": tokens,
                "ner_tags": ner_tags,
                "span_lengths": [len(pron_tokens)],
            })
    return examples


def tokenized_length_skip_filter(examples, tok, max_length):
    kept, skipped = [], 0
    for e in examples:
        enc = tok(e["tokens"], is_split_into_words=True, truncation=False)
        if len(enc["input_ids"]) > max_length:
            skipped += 1
            continue
        kept.append(e)
    return kept, skipped


variants_per_template = METHOD_A_VARIANTS_PRIMARY
method_a_raw = build_method_a(templates, variants_per_template, seed=AUG_SEED)
method_b_raw = build_method_b(SIMPLE_CARRIERS, METHOD_B_TOTAL, seed=AUG_SEED + 1)

method_a_kept, method_a_skipped = tokenized_length_skip_filter(method_a_raw, tokenizer, MAX_LENGTH)
method_b_kept, method_b_skipped = tokenized_length_skip_filter(method_b_raw, tokenizer, MAX_LENGTH)
total_skipped = method_a_skipped + method_b_skipped

if total_skipped > AUG_SKIP_RESAMPLE_THRESHOLD:
    flag(
        f"{total_skipped} s2l-augmented examples exceeded {MAX_LENGTH} subwords "
        f"(threshold {AUG_SKIP_RESAMPLE_THRESHOLD}) with {variants_per_template} variants/template. "
        f"Reducing Method A to {METHOD_A_VARIANTS_FALLBACK} variants/template and retrying (Method B unchanged)."
    )
    variants_per_template = METHOD_A_VARIANTS_FALLBACK
    method_a_raw = build_method_a(templates, variants_per_template, seed=AUG_SEED)
    method_a_kept, method_a_skipped = tokenized_length_skip_filter(method_a_raw, tokenizer, MAX_LENGTH)
    total_skipped = method_a_skipped + method_b_skipped

s2l_examples = method_a_kept + method_b_kept

all_span_lengths = [l for e in s2l_examples for l in e["span_lengths"]]
AUG_SPAN_MIN = int(min(all_span_lengths))
AUG_SPAN_MAX = int(max(all_span_lengths))
AUG_SPAN_AVG = float(np.mean(all_span_lengths))
AUG_SPAN_MEDIAN = float(np.median(all_span_lengths))

print(f"Method A: {len(method_a_kept)} examples from {len(templates)} templates x {variants_per_template} variants "
      f"(skipped {method_a_skipped})")
print(f"Method B: {len(method_b_kept)} examples from {len(SIMPLE_CARRIERS)} carrier phrases (skipped {method_b_skipped})")
print(f"Total s2l_augmented: {len(s2l_examples)}  (skipped for >{MAX_LENGTH} subwords: {total_skipped})")
print(f"s2l_aug span length stats: min={AUG_SPAN_MIN} max={AUG_SPAN_MAX} "
      f"avg={AUG_SPAN_AVG:.1f} median={AUG_SPAN_MEDIAN:.1f}")

Method A: 1480 examples from 185 templates x 8 variants (skipped 0)
Method B: 520 examples from 16 carrier phrases (skipped 0)
Total s2l_augmented: 2000  (skipped for >128 subwords: 0)
s2l_aug span length stats: min=3 max=25 avg=10.5 median=9.0


In [12]:
# Build an HF Dataset matching train_clean's column schema, then concatenate.
required_cols = raw_datasets["train_clean"].column_names
target_features = raw_datasets["train_clean"].features
print("train_clean columns:", required_cols)


def _make_record(e):
    rec = {"tokens": e["tokens"], "ner_tags": e["ner_tags"]}
    if "sentence_id" in required_cols:
        rec["sentence_id"] = e["sentence_id"]
    if "source" in required_cols:
        rec["source"] = "s2l_synthetic"
    for col in required_cols:
        if col not in rec:
            rec[col] = None
    return rec


s2l_aug_dataset = Dataset.from_list([_make_record(e) for e in s2l_examples])
s2l_aug_dataset = s2l_aug_dataset.select_columns(required_cols)
try:
    s2l_aug_dataset = s2l_aug_dataset.cast(target_features)
except Exception as e:
    flag(f"Could not cast s2l_aug_dataset to train_clean's feature schema ({e}); concatenation may fail.")

train_clean_plus_s2l = concatenate_datasets([raw_datasets["train_clean"], s2l_aug_dataset])
print(
    f"Final training set for BERT-2: {len(train_clean_plus_s2l)} "
    f"({len(raw_datasets['train_clean'])} clean + {len(s2l_aug_dataset)} s2l_aug)"
)

train_clean columns: ['sentence_id', 'tokens', 'ner_tags']


Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Final training set for BERT-2: 2185 (185 clean + 2000 s2l_aug)


## 5. Tokenizer & label alignment

In [13]:
def tokenize_and_align_labels(example):
    """Per-example (NOT batched) tokenization + label alignment.
    First subword of each word keeps the original label; continuation
    subwords get -100 so they're ignored by the loss and by metrics."""
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    word_ids = tokenized.word_ids()
    labels = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev_word_id:
            labels.append(example["ner_tags"][wid])
        else:
            labels.append(-100)
        prev_word_id = wid
    tokenized["labels"] = labels
    return tokenized


_SPLITS_TO_TOKENIZE = {
    "train_clean": raw_datasets["train_clean"],
    "train_noisy": raw_datasets["train_noisy"],
    "val": raw_datasets["val"],
    "test": raw_datasets["test"],
    "train_clean_plus_s2l": train_clean_plus_s2l,
}

tokenized_datasets = {
    split: ds.map(tokenize_and_align_labels, batched=False, remove_columns=ds.column_names)
    for split, ds in _SPLITS_TO_TOKENIZE.items()
}

# --- Alignment sanity check on a few examples from the new combined split ---
print("Alignment check (first 3 examples of train_clean_plus_s2l):")
for i in range(min(3, len(train_clean_plus_s2l))):
    orig = train_clean_plus_s2l[i]
    enc = tokenizer(orig["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = enc.word_ids()
    subwords = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    aligned = tokenized_datasets["train_clean_plus_s2l"][i]["labels"]
    pairs = [(sw, ID2LABEL.get(lab, "IGN")) for sw, lab in zip(subwords, aligned)]
    print(f"  ex {i}: {pairs}")
    non_ignored = [l for l in aligned if l != -100]
    assert len(non_ignored) == len(orig["ner_tags"]), "Alignment produced wrong number of labeled tokens"
print("Alignment check passed: exactly one labeled subtoken per original word.")

Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Map:   0%|          | 0/1261 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/2185 [00:00<?, ? examples/s]

Alignment check (first 3 examples of train_clean_plus_s2l):
  ex 0: [('[CLS]', 'IGN'), ('по', 'O'), ('свои', 'O'), ('##ству', 'IGN'), ('логариф', 'O'), ('##мов', 'IGN'), (',', 'IGN'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('произведения', 'I-MATH'), ('экспонен', 'I-MATH'), ('##ты', 'IGN'), ('в', 'I-MATH'), ('степени', 'I-MATH'), ('квадрата', 'I-MATH'), ('логариф', 'I-MATH'), ('##ма', 'IGN'), ('икс', 'I-MATH'), ('и', 'I-MATH'), ('игре', 'I-MATH'), ('##к', 'IGN'), ('распадается', 'O'), ('в', 'O'), ('сумму', 'O'), (',', 'IGN'), ('а', 'O'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('от', 'I-MATH'), ('произведения', 'I-MATH'), ('а', 'I-MATH'), ('и', 'I-MATH'), ('бэ', 'I-MATH'), ('равен', 'O'), ('сумме', 'O'), ('логариф', 'O'), ('##мов', 'IGN'), ('сом', 'O'), ('##но', 'IGN'), ('##жите', 'IGN'), ('##леи', 'IGN'), ('.', 'IGN'), ('[SEP]', 'IGN')]
  ex 1: [('[CLS]', 'IGN'), ('пусть', 'O'), ('даны', 'O'), ('множества', 'O'), ('a', 'O'), (',', 'IGN'), ('b', 'O'), (',', 'IGN'), ('w', 'O'), ('и', 'O'

## 6. Data collator & metrics

In [14]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


def _to_tag_sequences(predictions, label_ids):
    """Convert (predictions, label_ids) arrays into lists of string tag
    sequences per sentence, skipping -100 positions."""
    pred_ids = np.argmax(predictions, axis=2)
    true_seqs, pred_seqs = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        true_seq, pred_seq = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            true_seq.append(ID2LABEL[int(l)])
            pred_seq.append(ID2LABEL[int(p)])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    return true_seqs, pred_seqs


def compute_metrics(eval_pred):
    predictions, label_ids = eval_pred
    true_seqs, pred_seqs = _to_tag_sequences(predictions, label_ids)

    # Primary metric: span-level F1, strict IOB2 scheme (seqeval).
    span_f1 = seqeval_f1(true_seqs, pred_seqs, mode="strict", scheme=IOB2)

    # Token-level F1: binary, B-MATH+I-MATH vs O.
    flat_true = [t for seq in true_seqs for t in seq]
    flat_pred = [t for seq in pred_seqs for t in seq]
    bin_true = [0 if t == "O" else 1 for t in flat_true]
    bin_pred = [0 if t == "O" else 1 for t in flat_pred]
    token_f1 = sklearn_f1(bin_true, bin_pred, average="binary", zero_division=0)

    # Exact match: fraction of sentences with a perfectly correct BIO sequence.
    exact_matches = sum(1 for t, p in zip(true_seqs, pred_seqs) if t == p)
    exact_match = exact_matches / len(true_seqs) if true_seqs else 0.0

    return {"span_f1": span_f1, "token_f1": token_f1, "exact_match": exact_match}

## 7. Training utilities

`BestModelTracker` keeps the best-eval weights in **CPU memory** instead of
writing a full checkpoint to disk every epoch. In the previous run, writing
model shards to disk was ~90-95% of total wall-clock training time, so
`save_strategy="no"` is used for every stage here; the only disk write that
remains is the single, necessary hand-off of Stage-1 weights to Stage 2 in
BERT-3 (needed for a genuinely fresh optimizer).

In [15]:
# Support both old (`evaluation_strategy`) and new (`eval_strategy`) TrainingArguments APIs.
_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
_EVAL_KEY = "evaluation_strategy" if "evaluation_strategy" in _TA_PARAMS else "eval_strategy"

# Support both old (`tokenizer`) and new (`processing_class`) Trainer APIs.
_TRAINER_PARAMS = inspect.signature(Trainer.__init__).parameters
_TOKENIZER_KEY = "tokenizer" if "tokenizer" in _TRAINER_PARAMS else "processing_class"


def build_model(checkpoint_path=None):
    """Build a fresh model. If checkpoint_path is None, loads ACTIVE_MODEL_NAME
    from the Hub (with one fallback to FALLBACK_MODEL_NAME if that fails).
    If checkpoint_path is a local path (e.g. a Stage-1 checkpoint), loads from
    there directly -- no Hub fallback applies."""
    global ACTIVE_MODEL_NAME
    path = checkpoint_path or ACTIVE_MODEL_NAME
    try:
        model = AutoModelForTokenClassification.from_pretrained(
            path, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
        )
    except Exception as e:
        if checkpoint_path is None and ACTIVE_MODEL_NAME != FALLBACK_MODEL_NAME:
            flag(f"Model load for {path} failed: {e}. Falling back to {FALLBACK_MODEL_NAME}.")
            ACTIVE_MODEL_NAME = FALLBACK_MODEL_NAME
            model = AutoModelForTokenClassification.from_pretrained(
                ACTIVE_MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
            )
        else:
            raise
    return model


def all_zero_span_f1(log_history):
    vals = [h["eval_span_f1"] for h in log_history if "eval_span_f1" in h]
    return len(vals) > 0 and all(v == 0.0 for v in vals)


class BestModelTracker(TrainerCallback):
    """Keeps the best-eval state_dict in CPU memory (no disk I/O) and signals
    early stopping after `patience` evals with no improvement on `metric`.
    Replaces save_strategy='epoch' + load_best_model_at_end + EarlyStoppingCallback."""

    def __init__(self, model, metric="eval_span_f1", patience=None):
        self.model = model
        self.metric = metric
        self.patience = patience
        self.best_metric = None
        self.best_epoch = None
        self.best_state = None
        self.num_bad = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or self.metric not in metrics:
            return control
        value = metrics[self.metric]
        if self.best_metric is None or value > self.best_metric:
            self.best_metric = value
            self.best_epoch = state.epoch
            self.best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
            self.num_bad = 0
        else:
            self.num_bad += 1
            if self.patience is not None and self.num_bad >= self.patience:
                control.should_training_stop = True
        return control

    def restore_best(self):
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
        return self.best_epoch, self.best_metric


def train_one_stage(
    model,
    train_dataset,
    eval_dataset,
    output_dir,
    lr,
    epochs,
    batch_size=16,
    early_stopping_patience=None,
    label_smoothing_factor=0.0,
    seed=42,
    run_label="run",
):
    """Train one stage. Returns (trainer, elapsed_minutes, best_epoch, best_val_f1).
    best_epoch/best_val_f1 are None when eval_dataset/early_stopping_patience
    aren't both set (e.g. BERT-3 Stage 1). Handles CUDA OOM by retrying once
    with batch_size=8 + grad_accum=2 (keeps effective batch size at 16)."""
    do_eval = eval_dataset is not None and early_stopping_patience is not None

    def make_args(bsz, grad_accum):
        kwargs = dict(
            output_dir=output_dir,
            learning_rate=lr,
            num_train_epochs=epochs,
            per_device_train_batch_size=bsz,
            per_device_eval_batch_size=32,
            gradient_accumulation_steps=grad_accum,
            weight_decay=0.01,
            label_smoothing_factor=label_smoothing_factor,
            save_strategy="no",  # checkpointing handled in-memory by BestModelTracker
            fp16=(DEVICE == "cuda"),
            seed=seed,
            logging_steps=10,
            report_to="none",
            disable_tqdm=False,
        )
        kwargs[_EVAL_KEY] = "epoch" if do_eval else "no"
        return TrainingArguments(**kwargs)

    bsz, grad_accum = batch_size, 1
    start = time.time()
    attempt = 0
    tracker = None
    while True:
        attempt += 1
        args = make_args(bsz, grad_accum)
        tracker = BestModelTracker(model, metric="eval_span_f1", patience=early_stopping_patience) if do_eval else None
        trainer_kwargs = dict(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset if do_eval else None,
            data_collator=data_collator,
            compute_metrics=compute_metrics if do_eval else None,
            callbacks=[tracker] if tracker else [],
        )
        trainer_kwargs[_TOKENIZER_KEY] = tokenizer
        trainer = Trainer(**trainer_kwargs)
        try:
            trainer.train()
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            gc.collect()
            if attempt >= 2:
                raise
            flag(
                f"[{run_label}] CUDA OOM with batch_size={bsz}. Retrying with "
                f"batch_size=8, gradient_accumulation_steps=2 (effective batch size unchanged)."
            )
            bsz, grad_accum = 8, 2

    elapsed_min = (time.time() - start) / 60.0

    if elapsed_min > 30:
        flag(f"[{run_label}] took {elapsed_min:.1f} min (> 30 min). Continuing per protocol.")

    if do_eval and all_zero_span_f1(trainer.state.log_history):
        raise RuntimeError(
            f"[{run_label}] val span-F1 is exactly 0.0 for ALL epochs. "
            f"This indicates a label-alignment or metric-computation bug. Stopping per protocol "
            f"instead of continuing -- inspect tokenize_and_align_labels / compute_metrics before re-running."
        )

    best_epoch, best_val_f1 = (None, None)
    if tracker is not None:
        best_epoch, best_val_f1 = tracker.restore_best()  # loads best-eval weights back into `model` in place

    return trainer, elapsed_min, best_epoch, best_val_f1


def cleanup_dir(path):
    shutil.rmtree(path, ignore_errors=True)

## 8. BERT-2: clean + s2l augmented (no curriculum)

Can be run independently of BERT-3 (Section 9) as long as Sections 1-7 have
been executed in this kernel session.

In [16]:
bert2_results = []  # one dict per seed
bert2_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}
bert2_test_predictions = {}  # seed -> (true_seqs, pred_seqs), first-subword level

for seed in SEEDS:
    print(f"\n=== BERT-2 | seed={seed} ===")
    set_seed(seed)
    model = build_model()
    out_dir = f"./tmp_bert2_seed{seed}"

    trainer, elapsed_min, best_epoch, best_val_f1 = train_one_stage(
        model=model,
        train_dataset=tokenized_datasets["train_clean_plus_s2l"],
        eval_dataset=tokenized_datasets["val"],
        output_dir=out_dir,
        lr=2e-5,
        epochs=10,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing_factor=0.0,
        seed=seed,
        run_label=f"BERT-2 seed{seed}",
    )

    val_metrics = trainer.evaluate(tokenized_datasets["val"])
    test_metrics = trainer.evaluate(tokenized_datasets["test"])

    # Keep the actual test-set predictions (not just aggregate metrics) for the
    # error-analysis section at the end of the notebook.
    test_pred_output = trainer.predict(tokenized_datasets["test"])
    true_seqs_i, pred_seqs_i = _to_tag_sequences(test_pred_output.predictions, test_pred_output.label_ids)
    bert2_test_predictions[seed] = (true_seqs_i, pred_seqs_i)

    if best_val_f1 is not None and best_val_f1 > bert2_best_overall["val_span_f1"]:
        bert2_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    bert2_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["eval_span_f1"],
        "test_span_f1": test_metrics["eval_span_f1"],
        "test_token_f1": test_metrics["eval_token_f1"],
        "test_exact_match": test_metrics["eval_exact_match"],
        "train_time_min": elapsed_min,
    })
    print(bert2_results[-1])

    cleanup_dir(out_dir)
    del trainer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


=== BERT-2 | seed=42 ===


pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

...

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

Epoch,Training Loss,Validation Loss,Span F1,Token F1,Exact Match
1,0.064733,1.237039,0.125000,0.782201,0.066667
2,0.017824,1.396541,0.203822,0.793881,0.066667
3,0.004995,1.766991,0.152866,0.779503,0.066667
4,0.002197,1.863466,0.133333,0.781395,0.066667
5,0.006613,1.869240,0.177215,0.785266,0.066667


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.006613,1.396541,5,0.203822,0.793881,0.066667


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.006613,1.408816,5,0.161290,0.809806,0.120000


{'seed': 456, 'val_span_f1': 0.2038216560509554, 'test_span_f1': 0.16129032258064516, 'test_token_f1': 0.8098055790363483, 'test_exact_match': 0.12, 'train_time_min': 1.1494749943415323}


## 9. BERT-3: curriculum (Mathematicon → clean + s2l augmented)

Can be run independently of BERT-2 (Section 8) as long as Sections 1-7 have
been executed in this kernel session.

In [17]:
bert3_results = []
bert3_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}
bert3_test_predictions = {}  # seed -> (true_seqs, pred_seqs), first-subword level

for seed in SEEDS:
    print(f"\n=== BERT-3 | seed={seed} | Stage 1 (noisy) ===")
    set_seed(seed)
    model = build_model()
    stage1_dir = f"./tmp_bert3_stage1_seed{seed}"

    trainer_s1, stage1_min, _, _ = train_one_stage(
        model=model,
        train_dataset=tokenized_datasets["train_noisy"],
        eval_dataset=None,
        output_dir=stage1_dir,
        lr=3e-5,
        epochs=3,
        batch_size=16,
        early_stopping_patience=None,
        label_smoothing_factor=0.1,
        seed=seed,
        run_label=f"BERT-3 seed{seed} stage1",
    )

    # Save Stage-1 weights to disk so Stage 2 loads a genuinely fresh
    # model + fresh optimizer/Trainer (no carried-over optimizer state).
    # This is the one necessary disk write per seed -- everything else in
    # this notebook keeps checkpoints in memory (see BestModelTracker).
    stage1_ckpt = f"./tmp_bert3_stage1_ckpt_seed{seed}"
    trainer_s1.save_model(stage1_ckpt)
    cleanup_dir(stage1_dir)
    del trainer_s1, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"=== BERT-3 | seed={seed} | Stage 2 (clean+s2l, fresh optimizer) ===")
    model_s2 = build_model(checkpoint_path=stage1_ckpt)
    stage2_dir = f"./tmp_bert3_stage2_seed{seed}"

    trainer_s2, stage2_min, best_epoch, best_val_f1 = train_one_stage(
        model=model_s2,
        train_dataset=tokenized_datasets["train_clean_plus_s2l"],
        eval_dataset=tokenized_datasets["val"],
        output_dir=stage2_dir,
        lr=1.5e-5,
        epochs=15,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing_factor=0.0,
        seed=seed,
        run_label=f"BERT-3 seed{seed} stage2",
    )

    val_metrics = trainer_s2.evaluate(tokenized_datasets["val"])
    test_metrics = trainer_s2.evaluate(tokenized_datasets["test"])

    test_pred_output = trainer_s2.predict(tokenized_datasets["test"])
    true_seqs_i, pred_seqs_i = _to_tag_sequences(test_pred_output.predictions, test_pred_output.label_ids)
    bert3_test_predictions[seed] = (true_seqs_i, pred_seqs_i)

    if best_val_f1 is not None and best_val_f1 > bert3_best_overall["val_span_f1"]:
        bert3_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    total_min = stage1_min + stage2_min

    bert3_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["eval_span_f1"],
        "test_span_f1": test_metrics["eval_span_f1"],
        "test_token_f1": test_metrics["eval_token_f1"],
        "test_exact_match": test_metrics["eval_exact_match"],
        "stage1_time_min": stage1_min,
        "stage2_time_min": stage2_min,
        "train_time_min": total_min,
    })
    print(bert3_results[-1])

    cleanup_dir(stage2_dir)
    cleanup_dir(stage1_ckpt)
    del trainer_s2, model_s2
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


=== BERT-3 | seed=42 | Stage 1 (noisy) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

Step,Training Loss
10,0.908858
20,0.756277
30,0.695984
40,0.682405
50,0.629148
60,0.578290
70,0.565270
80,0.536142
90,0.486019
100,0.488482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Span F1,Token F1,Exact Match
1,0.085517,1.425929,0.143791,0.770865,0.033333
2,0.020686,1.377147,0.115607,0.794338,0.066667
3,0.010910,1.760123,0.137143,0.782540,0.066667
4,0.007830,1.839106,0.173913,0.787015,0.066667
5,0.007948,1.928649,0.142857,0.790064,0.066667
6,0.002229,2.051364,0.148148,0.785546,0.066667
7,0.002209,1.873847,0.150000,0.801972,0.066667


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.002209,1.839106,7,0.173913,0.787015,0.066667


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.002209,1.834016,7,0.133858,0.797841,0.040000


{'seed': 456, 'val_span_f1': 0.17391304347826086, 'test_span_f1': 0.13385826771653545, 'test_token_f1': 0.7978414279784143, 'test_exact_match': 0.04, 'stage1_time_min': 0.35626724163691204, 'stage2_time_min': 1.584638198216756, 'train_time_min': 1.940905439853668}


## 10. Aggregate & report

Builds the final `<executor_output id="E4b">` block. BERT-0 / BERT-1 numbers
are hardcoded (not rerun), per protocol.

In [18]:
BERT0_TEST_SPAN_F1 = (0.1932, 0.0043)
BERT1_TEST_SPAN_F1 = (0.1844, 0.0177)
BERT1_DELTA_VS_BERT0 = -0.0088


def _mean_std(results, key):
    vals = [r[key] for r in results]
    return float(np.mean(vals)), float(np.std(vals))


def _fmt(mean, std):
    return f"{mean:.4f}±{std:.4f}"


def _fmt_delta(delta):
    sign = "+" if delta >= 0 else ""
    return f"{sign}{delta:.4f}"


def _results_table(results, model_name):
    lines = [
        "| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match | Train Time (min) |",
        "|------|-------------|--------------|----------------|-------------------|-------------------|",
    ]
    for r in results:
        time_key = "train_time_min"
        lines.append(
            f"| {r['seed']} | {r['val_span_f1']:.4f} | {r['test_span_f1']:.4f} | "
            f"{r['test_token_f1']:.4f} | {r['test_exact_match']:.4f} | {int(round(r[time_key]))} |"
        )
    span_mean, span_std = _mean_std(results, "test_span_f1")
    token_mean, token_std = _mean_std(results, "test_token_f1")
    em_mean, em_std = _mean_std(results, "test_exact_match")
    lines.append(
        f"| **Mean±Std** | — | **{_fmt(span_mean, span_std)}** | {_fmt(token_mean, token_std)} | "
        f"{_fmt(em_mean, em_std)} | — |"
    )
    return "\n".join(lines), span_mean, span_std


bert2_table, bert2_span_mean, bert2_span_std = _results_table(bert2_results, "BERT-2")
bert3_table, bert3_span_mean, bert3_span_std = _results_table(bert3_results, "BERT-3")

bert2_delta = bert2_span_mean - BERT0_TEST_SPAN_F1[0]
bert3_delta = bert3_span_mean - BERT0_TEST_SPAN_F1[0]

n_clean = len(raw_datasets["train_clean"])
n_s2l_aug = len(s2l_aug_dataset)

bert2_time_per_seed = int(round(np.mean([r["train_time_min"] for r in bert2_results])))
bert3_time_per_seed = int(round(np.mean([r["train_time_min"] for r in bert3_results])))
bert3_stage1_per_seed = int(round(np.mean([r["stage1_time_min"] for r in bert3_results])))
bert3_stage2_per_seed = int(round(np.mean([r["stage2_time_min"] for r in bert3_results])))


def _best_ckpt_line(best):
    epoch_str = f"{best['epoch']:.1f}" if best["epoch"] is not None else "N/A"
    f1_str = f"{best['val_span_f1']:.4f}" if best["val_span_f1"] is not None else "N/A"
    return f"Best checkpoint: seed={best['seed']}, epoch={epoch_str}, val_span_f1={f1_str}"


bert2_best_line = _best_ckpt_line(bert2_best_overall)
bert3_best_line = _best_ckpt_line(bert3_best_overall)

span_improved = bert2_delta > 0 or bert3_delta > 0
notes_obs = []
if span_improved:
    notes_obs.append("s2l augmentation improved test span-F1 over BERT-0/BERT-1, consistent with the "
                      "span-length-mismatch hypothesis (training spans now better match the real test distribution).")
else:
    notes_obs.append("s2l augmentation did not improve test span-F1 over BERT-0/BERT-1 in this run; "
                      "span boundary detection remains the dominant error mode despite the closer length match.")
if variants_per_template != METHOD_A_VARIANTS_PRIMARY:
    notes_obs.append(f"Method A fell back to {variants_per_template} variants/template after exceeding the "
                      f"skip threshold at {METHOD_A_VARIANTS_PRIMARY}.")
if RUN_FLAGS:
    notes_obs.append(f"{len(RUN_FLAGS)} flag(s) raised during the run (see RUN_FLAGS): " + " | ".join(RUN_FLAGS))
notes_line = "Notes: " + " ".join(notes_obs)

executor_output = f"""<executor_output id="E4b">

## ruBERT Iteration 2 Results

### Augmentation Stats

- s2l pronunciations loaded: {n_s2l_loaded}
- s2l pronunciations after filtering (3-25 tokens): {n_s2l_filtered}
- Method A (template injection): {len(method_a_kept)} examples from {len(templates)} templates x {variants_per_template} variants/template
- Method B (carrier phrases): {len(method_b_kept)} examples from {len(SIMPLE_CARRIERS)} carrier phrases
- Total s2l_augmented examples: {len(s2l_examples)}
- Skipped for exceeding {MAX_LENGTH} subword tokens: {total_skipped}
- Final BERT-2 training set size: {len(train_clean_plus_s2l)} ({n_clean} clean + {n_s2l_aug} s2l_aug)
- s2l_aug span length stats: min={AUG_SPAN_MIN} max={AUG_SPAN_MAX} avg={AUG_SPAN_AVG:.4f} median={AUG_SPAN_MEDIAN:.4f}

### BERT-2: Clean + s2l augmented

{bert2_table}

{bert2_best_line}
Training time per seed (approx): {bert2_time_per_seed} min

### BERT-3: Curriculum (Mathematicon → clean + s2l)

{bert3_table}

{bert3_best_line}
Training time per seed (approx): {bert3_time_per_seed} min (stage1: {bert3_stage1_per_seed} min, stage2: {bert3_stage2_per_seed} min)

### Full Comparison

| Model  | Train Data | Test Span-F1 (mean±std) | Δ vs BERT-0 |
|--------|-----------|-------------------------|-------------|
| BERT-0 | {n_clean} clean | {_fmt(*BERT0_TEST_SPAN_F1)} | — |
| BERT-1 | curriculum: 1261 noisy → {n_clean} clean | {_fmt(*BERT1_TEST_SPAN_F1)} | {_fmt_delta(BERT1_DELTA_VS_BERT0)} |
| BERT-2 | {n_clean} clean + ~{n_s2l_aug} s2l_aug | {_fmt(bert2_span_mean, bert2_span_std)} | {_fmt_delta(bert2_delta)} |
| BERT-3 | curriculum: 1261 noisy → {n_clean} clean + ~{n_s2l_aug} s2l_aug | {_fmt(bert3_span_mean, bert3_span_std)} | {_fmt_delta(bert3_delta)} |

{notes_line}

</executor_output>"""

print(executor_output)

<executor_output id="E4b">

## ruBERT Iteration 2 Results

### Augmentation Stats

- s2l pronunciations loaded: 71108
- s2l pronunciations after filtering (3-25 tokens): 59954
- Method A (template injection): 1480 examples from 185 templates x 8 variants/template
- Method B (carrier phrases): 520 examples from 16 carrier phrases
- Total s2l_augmented examples: 2000
- Skipped for exceeding 128 subword tokens: 0
- Final BERT-2 training set size: 2185 (185 clean + 2000 s2l_aug)
- s2l_aug span length stats: min=3 max=25 avg=10.5198 median=9.0000

### BERT-2: Clean + s2l augmented

| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match | Train Time (min) |
|------|-------------|--------------|----------------|-------------------|-------------------|
| 42 | 0.2000 | 0.1673 | 0.8118 | 0.1000 | 1 |
| 123 | 0.1905 | 0.1617 | 0.8093 | 0.0800 | 2 |
| 456 | 0.2038 | 0.1613 | 0.8098 | 0.1200 | 1 |
| **Mean±Std** | — | **0.1634±0.0028** | 0.8103±0.0011 | 0.1000±0.0163 | — |

Best che

## 11. Error analysis: test-set predictions vs gold

Prints, for the best-val-F1 seed of each model, every test sentence with its
gold and predicted BIO tags (first-subword level, same positions the metrics
above are computed on), then an errors-only view showing exactly which
spans were missed, invented, or boundary-shifted.

In [20]:
def extract_spans(tags):
    """Contiguous B-MATH(I-MATH)* runs -> list of (start, end_exclusive)."""
    spans = []
    start = None
    for i, t in enumerate(tags):
        if t == "B-MATH":
            if start is not None:
                spans.append((start, i))
            start = i
        elif t == "I-MATH":
            if start is None:
                start = i  # malformed (I-MATH without a preceding B-MATH); defensive only
        else:
            if start is not None:
                spans.append((start, i))
                start = None
    if start is not None:
        spans.append((start, len(tags)))
    return spans


def render_sentence(tokens, tags):
    return " ".join(f"{tok}/{tag}" if tag != "O" else tok for tok, tag in zip(tokens, tags))


def span_text(tokens, span):
    s, e = span
    return " ".join(tokens[s:e])


def print_error_analysis(model_name, true_seqs, pred_seqs, tokens_list):
    print(f"\n{'='*90}\n{model_name}: test-set predictions ({len(true_seqs)} sentences)\n{'='*90}")

    n_exact = 0
    error_rows = []
    for i, (full_tokens, gold, pred) in enumerate(zip(tokens_list, true_seqs, pred_seqs)):
        # Align raw tokens to the (possibly truncated, first-subword-only) label
        # sequence: truncation only ever drops trailing words, so slicing to
        # len(gold) keeps the correspondence correct even for long sentences.
        tokens = full_tokens[:len(gold)]
        assert len(tokens) == len(gold) == len(pred), (
            f"Length mismatch at test sentence {i}: tokens={len(tokens)} gold={len(gold)} pred={len(pred)}"
        )
        if len(full_tokens) != len(gold):
            flag(f"{model_name}: test sentence {i} was truncated for evaluation "
                 f"({len(full_tokens)} raw tokens -> {len(gold)} labeled); "
                 f"showing only the retained prefix below.")
        is_exact = gold == pred
        if is_exact:
            n_exact += 1
        print(f"\n[{i}] GOLD: {render_sentence(tokens, gold)}")
        print(f"[{i}] PRED: {render_sentence(tokens, pred)}")
        if not is_exact:
            gold_spans = extract_spans(gold)
            pred_spans = extract_spans(pred)
            gold_set, pred_set = set(gold_spans), set(pred_spans)
            missed = gold_set - pred_set    # gold spans with no exact predicted match
            extra = pred_set - gold_set     # predicted spans with no exact gold match
            error_rows.append((i, tokens, gold_spans, pred_spans, missed, extra))
            print(f"[{i}] MISMATCH -- gold: {[span_text(tokens, s) for s in gold_spans]} "
                  f"| pred: {[span_text(tokens, s) for s in pred_spans]}")

    n_total = len(true_seqs)
    n_missed_total = sum(len(r[4]) for r in error_rows)
    n_extra_total = sum(len(r[5]) for r in error_rows)

    print(f"\n--- {model_name}: errors-only summary ---")
    print(f"Exact-match sentences: {n_exact}/{n_total} ({n_exact/n_total:.4f})")
    print(f"Sentences with >=1 span error: {len(error_rows)}/{n_total}")
    print(f"Missed gold spans (false negatives, no exact predicted match): {n_missed_total}")
    print(f"Extra predicted spans (false positives, no exact gold match): {n_extra_total}")

    print(f"\n--- {model_name}: detailed error list ---")
    if not error_rows:
        print("(no errors -- every test sentence matched exactly)")
    for i, tokens, gold_spans, pred_spans, missed, extra in error_rows:
        print(f"\n[{i}] sentence: {' '.join(tokens)}")
        print(f"    gold spans:      {[span_text(tokens, s) for s in gold_spans]}")
        print(f"    predicted spans: {[span_text(tokens, s) for s in pred_spans]}")
        if missed:
            print(f"    MISSED (gold, no exact prediction):  {[span_text(tokens, s) for s in missed]}")
        if extra:
            print(f"    EXTRA (predicted, not in gold):      {[span_text(tokens, s) for s in extra]}")

    return {"n_exact": n_exact, "n_total": n_total, "n_missed": n_missed_total, "n_extra": n_extra_total}


test_tokens = raw_datasets["test"]["tokens"]

bert2_err_seed = bert2_best_overall["seed"]
bert3_err_seed = bert3_best_overall["seed"]

if bert2_err_seed in bert2_test_predictions:
    true2, pred2 = bert2_test_predictions[bert2_err_seed]
    bert2_error_summary = print_error_analysis(
        f"BERT-2 (best seed={bert2_err_seed})", true2, pred2, test_tokens
    )
else:
    flag(f"No stored test predictions for BERT-2 seed {bert2_err_seed}; skipping its error analysis.")

if bert3_err_seed in bert3_test_predictions:
    true3, pred3 = bert3_test_predictions[bert3_err_seed]
    bert3_error_summary = print_error_analysis(
        f"BERT-3 (best seed={bert3_err_seed})", true3, pred3, test_tokens
    )
else:
    flag(f"No stored test predictions for BERT-3 seed {bert3_err_seed}; skipping its error analysis.")


BERT-2 (best seed=456): test-set predictions (50 sentences)
[FLAG] BERT-2 (best seed=456): test sentence 0 was truncated for evaluation (112 raw tokens -> 104 labeled); showing only the retained prefix below.

[0] GOLD: Итак, число B называется пределом/B-MATH функций/I-MATH х/I-MATH от/I-MATH двух/I-MATH переменных/I-MATH х/I-MATH и/I-MATH грек/I-MATH в/I-MATH точке/I-MATH A/I-MATH . Если существует/B-MATH такое/I-MATH дельта/I-MATH , только все-таки начнем с другого. Если/B-MATH для/I-MATH любого/I-MATH эпселен/I-MATH большего/I-MATH нуля,/I-MATH существует/I-MATH такое/I-MATH дельта/I-MATH больше/I-MATH нуля,/I-MATH то/I-MATH для/I-MATH всех/I-MATH точек/I-MATH M/I-MATH ,/I-MATH принадлежащих/I-MATH области/I-MATH определения, это очень важно. Точно так же, как и в случае одной перемены. Мы там тоже говорили о том, что должно принадлежать области определения. Таких,/B-MATH что/I-MATH расстояние/I-MATH от/I-MATH точки/I-MATH M/I-MATH до/I-MATH точки/I-MATH A/I-MATH меньше/I-MATH дел